# HW4P2: Automatic Speech Recognition with an Encoder-Decoder Transformer

**Deadlines:**
- Checkpoint Submission (CER ≤ 50%): April 10, 2026 @ 11:59PM EST
- Kaggle Final Submission: April 24, 2026 @ 11:59PM EST
- Code Submission (Autolab): April 26, 2026

# Setup

In [ ]:
# Step 1: Clone / pull your repo
import os
os.environ["GITHUB_TOKEN"] = "YOUR_GITHUB_TOKEN_HERE"  # TODO: Set your token
GITHUB_USERNAME = "yufeisong0914"
REPO_NAME       = "IDL-HW4"
TOKEN = os.environ.get("GITHUB_TOKEN")
repo_url = f"https://{TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
!git clone {repo_url}


In [ ]:
# Pull latest changes (run from inside repo dir)
!cd {REPO_NAME} && git pull

In [ ]:
# Kaggle API authentication — re-run this after every kernel restart
import os, json

KAGGLE_USERNAME = "yufeison"
KAGGLE_KEY      = "KGAT_0e2401c4d351c4ce89b30f0778b2f306"

os.makedirs("/root/.config/kaggle", exist_ok=True)
with open("/root/.config/kaggle/kaggle.json", "w") as kf:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, kf)
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

# Re-import to pick up new credentials
import importlib, kaggle as _kg
importlib.reload(_kg)
import kaggle
api = kaggle.api
print("Kaggle API ready")


In [ ]:
# Kaggle 已预装 torch/tokenizers 等主要包，只安装缺少的
!pip install -q torchmetrics torchinfo wandb
# 安装完成后重启 kernel（点 Run > Restart & Clear Outputs）


In [ ]:
# Step 4: Move into the repo directory
import os
os.chdir('IDL-HW4')
!ls

# Imports

In [ ]:
from hw4lib.data import (
    H4Tokenizer,
    ASRDataset,
    verify_dataloader
)
from hw4lib.model import (
    DecoderOnlyTransformer,
    EncoderDecoderTransformer
)
from hw4lib.utils import (
    create_scheduler,
    create_optimizer,
    plot_lr_schedule
)
from hw4lib.trainers import (
    ASRTrainer,
    ProgressiveTrainer
)
from torch.utils.data import DataLoader
import yaml
import gc
import torch
from torchinfo import summary
import os
import json
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Implementation Tests
Run these before training to confirm all implementations pass.

In [ ]:
!python -m tests.test_dataset_asr

In [ ]:
!python -m tests.test_sublayer_crossattention

In [ ]:
!python -m tests.test_decoderlayer_crossattention

In [ ]:
!python -m tests.test_encoderlayer_selfattention

In [ ]:
!python -m tests.test_transformer_encoder_decoder

In [ ]:
!python -m tests.test_decoding --mode beam

# Config

In [ ]:
%%writefile config.yaml

Name                      : "yufeison-carnegie-mellon-university"

###### Tokenization ------------------------------------------------------------
tokenization:
  token_type                : "char"       # [char, 1k, 5k, 10k]
  token_map :
      'char': 'hw4lib/data/tokenizer_jsons/tokenizer_char.json'
      '1k'  : 'hw4lib/data/tokenizer_jsons/tokenizer_1000.json'
      '5k'  : 'hw4lib/data/tokenizer_jsons/tokenizer_5000.json'
      '10k' : 'hw4lib/data/tokenizer_jsons/tokenizer_10000.json'

###### Dataset -----------------------------------------------------------------
data:
  root                 : "/kaggle/input/s26-11785-hw4-data/hw4p2_data"  # TODO: Set data root
  train_partition      : "train-clean-100"
  val_partition        : "dev-clean"
  test_partition       : "test-clean"
  subset_size          : null
  batch_size           : 64
  NUM_WORKERS          : 4
  num_feats            : 80
  norm                 : "global_mvn"
  specaug              : True
  specaug_conf:
    apply_freq_mask        : True
    apply_time_mask        : True
    num_freq_mask          : 2
    num_time_mask          : 2
    freq_mask_width_range  : 27
    time_mask_width_range  : 100

###### Network Specs -----------------------------------------------------------
model:
  input_dim            : 80
  time_reduction       : 4
  reduction_method     : "conv"
  num_encoder_layers   : 4
  num_encoder_heads    : 4
  d_ff_encoder         : 1024
  num_decoder_layers   : 2
  num_decoder_heads    : 4
  d_ff_decoder         : 1024
  d_model              : 256
  dropout              : 0.1
  weight_tying         : False
  layer_drop_rate      : 0.0
  skip_encoder_pe      : False
  skip_decoder_pe      : False

###### Training ---------------------------------------------------------------
training:
  use_wandb                   : False
  wandb_run_id                : "none"
  resume                      : False
  epochs                      : 30
  gradient_accumulation_steps : 2
  wandb_project               : "hw4p2"

###### Loss -------------------------------------------------------------------
loss:
  label_smoothing : 0.1
  ctc_weight      : 0.1

###### Optimizer --------------------------------------------------------------
optimizer:
  name         : "adamw"
  lr           : 3.0e-4
  weight_decay : 0.01
  param_groups : []
  layer_decay:
    enabled    : False
    decay_rate : 0.75
  sgd:
    momentum  : 0.9
    nesterov  : True
    dampening : 0
  adam:
    betas   : [0.9, 0.999]
    eps     : 1.0e-8
    amsgrad : False
  adamw:
    betas   : [0.9, 0.98]
    eps     : 1.0e-9
    amsgrad : False

###### Scheduler --------------------------------------------------------------
scheduler:
  name: "cosine"
  reduce_lr:
    mode           : "min"
    factor         : 0.5
    patience       : 5
    threshold      : 0.0001
    threshold_mode : "rel"
    cooldown       : 2
    min_lr         : 1.0e-7
    eps            : 1.0e-8
  cosine:
    T_max      : 30
    eta_min    : 1.0e-7
    last_epoch : -1
  cosine_warm:
    T_0        : 10
    T_mult     : 2
    eta_min    : 1.0e-7
    last_epoch : -1
  warmup:
    enabled      : True
    type         : "linear"
    epochs       : 3
    start_factor : 0.1
    end_factor   : 1.0

In [ ]:
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)
print(json.dumps(config, indent=2))

# Tokenizer

In [ ]:
Tokenizer = H4Tokenizer(
    token_map  = config['tokenization']['token_map'],
    token_type = config['tokenization']['token_type']
)
print(f"Vocab size : {Tokenizer.vocab_size}")
print(f"SOS id     : {Tokenizer.sos_id}")
print(f"EOS id     : {Tokenizer.eos_id}")
print(f"PAD id     : {Tokenizer.pad_id}")

# Datasets

In [ ]:
train_dataset = ASRDataset(
    partition       = config['data']['train_partition'],
    config          = config['data'],
    tokenizer       = Tokenizer,
    isTrainPartition= True,
    global_stats    = None
)

# Carry computed global stats to val/test
global_stats = None
if config['data']['norm'] == 'global_mvn':
    global_stats = (train_dataset.global_mean, train_dataset.global_std)
    print("Global stats computed from training set.")

val_dataset = ASRDataset(
    partition       = config['data']['val_partition'],
    config          = config['data'],
    tokenizer       = Tokenizer,
    isTrainPartition= False,
    global_stats    = global_stats
)

test_dataset = ASRDataset(
    partition       = config['data']['test_partition'],
    config          = config['data'],
    tokenizer       = Tokenizer,
    isTrainPartition= False,
    global_stats    = global_stats
)

print(f"Train size : {len(train_dataset)}")
print(f"Val   size : {len(val_dataset)}")
print(f"Test  size : {len(test_dataset)}")

# Dataloaders

In [ ]:
train_loader = DataLoader(
    dataset    = train_dataset,
    batch_size = config['data']['batch_size'],
    shuffle    = True,
    num_workers= config['data']['NUM_WORKERS'] if device == 'cuda' else 0,
    pin_memory = True,
    collate_fn = train_dataset.collate_fn
)

val_loader = DataLoader(
    dataset    = val_dataset,
    batch_size = config['data']['batch_size'],
    shuffle    = False,
    num_workers= config['data']['NUM_WORKERS'] if device == 'cuda' else 0,
    pin_memory = True,
    collate_fn = val_dataset.collate_fn
)

test_loader = DataLoader(
    dataset    = test_dataset,
    batch_size = config['data']['batch_size'],
    shuffle    = False,
    num_workers= config['data']['NUM_WORKERS'] if device == 'cuda' else 0,
    pin_memory = True,
    collate_fn = test_dataset.collate_fn
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

In [ ]:
verify_dataloader(train_loader)
verify_dataloader(val_loader)
verify_dataloader(test_loader)

# Compute Max Lengths

In [ ]:
max_feat_len       = max(train_dataset.feat_max_len, val_dataset.feat_max_len, test_dataset.feat_max_len)
max_transcript_len = max(train_dataset.text_max_len, val_dataset.text_max_len)
max_len            = max(max_feat_len, max_transcript_len)

print("=" * 50)
print(f"{'Max Feature Length':<30} : {max_feat_len}")
print(f"{'Max Transcript Length':<30} : {max_transcript_len}")
print(f"{'Overall Max Length':<30} : {max_len}")
print("=" * 50)

# (Optional) WandB Login

In [ ]:
# Uncomment and set your key to enable WandB logging
# import wandb
# wandb.login(key="your_wandb_api_key_here")
# config['training']['use_wandb'] = True

# Training — Cold Start

Build model, trainer, optimizer, scheduler and train.

In [ ]:
# Build model
model_config = config['model'].copy()
model_config.update({
    'max_len'    : max_len,
    'num_classes': Tokenizer.vocab_size
})
model = EncoderDecoderTransformer(**model_config)

# Verify parameter count (must be < 30M)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")
assert total_params < 30_000_000, f"Parameter count {total_params} exceeds 30M limit!"

# Quick model summary on a sample batch
for batch in train_loader:
    padded_feats, padded_shifted, padded_golden, feat_lengths, transcript_lengths = batch
    break
model_stats = summary(
    model,
    input_data=[padded_feats, padded_shifted, feat_lengths, transcript_lengths]
)
print(model_stats)

In [ ]:
# Create trainer
RUN_NAME = "hw4p2-baseline"

trainer = ASRTrainer(
    model       = model,
    tokenizer   = Tokenizer,
    config      = config,
    run_name    = RUN_NAME,
    config_file = "config.yaml",
    device      = device
)

In [ ]:
# (Optional) Resume from checkpoint
# trainer.load_checkpoint('checkpoint-best-metric-model.pth')

In [ ]:
# Set up optimizer
trainer.optimizer = create_optimizer(
    model      = model,
    opt_config = config['optimizer']
)

In [ ]:
# Preview LR schedule
test_scheduler = create_scheduler(
    optimizer                  = trainer.optimizer,
    scheduler_config           = config['scheduler'],
    train_loader               = train_loader,
    gradient_accumulation_steps= config['training']['gradient_accumulation_steps']
)
plot_lr_schedule(
    scheduler                  = test_scheduler,
    num_epochs                 = config['training']['epochs'],
    train_loader               = train_loader,
    gradient_accumulation_steps= config['training']['gradient_accumulation_steps']
)

In [ ]:
# Set up scheduler for real training
trainer.scheduler = create_scheduler(
    optimizer                  = trainer.optimizer,
    scheduler_config           = config['scheduler'],
    train_loader               = train_loader,
    gradient_accumulation_steps= config['training']['gradient_accumulation_steps']
)

In [ ]:
# Train!
trainer.train(train_loader, val_loader, epochs=config['training']['epochs'])

# Inference & Kaggle Submission

In [ ]:
import kaggle
api = kaggle.api

# Load best checkpoint before inference
trainer.load_checkpoint('checkpoint-best-metric-model.pth')

In [ ]:
# Greedy decoding on test set
recognition_config = {
    'num_batches'   : None,
    'temperature'   : 1.0,
    'repeat_penalty': 1.0,
    'lm_weight'     : 0.0,
    'lm_model'      : None,
    'beam_width'    : 1,      # greedy
}

results = trainer.recognize(
    test_loader,
    recognition_config,
    config_name = "test_greedy",
    max_length  = max_transcript_len
)

generated = [r['generated'] for r in results]
results_df = pd.DataFrame({'id': range(len(generated)), 'transcription': generated})
results_df.to_csv("results.csv", index=False)
print(f"Saved {len(results_df)} predictions to results.csv")
results_df.head()

In [ ]:
# Submit to Kaggle
api.competition_submit(
    file_name   = "results.csv",
    message     = "HW4P2 greedy submission",
    competition = "11785-hw-4-p-2-asr-with-transformer-spring-2026"
)

## (Optional) Beam Search Submission

In [ ]:
# Beam search decoding (higher quality, slower)
beam_config = {
    'num_batches'   : None,
    'temperature'   : 1.0,
    'repeat_penalty': 1.0,
    'lm_weight'     : 0.0,
    'lm_model'      : None,
    'beam_width'    : 10,
}

beam_results = trainer.recognize(
    test_loader,
    beam_config,
    config_name = "test_beam10",
    max_length  = max_transcript_len
)

beam_generated = [r['generated'] for r in beam_results]
beam_df = pd.DataFrame({'id': range(len(beam_generated)), 'transcription': beam_generated})
beam_df.to_csv("results_beam10.csv", index=False)
print(f"Saved {len(beam_df)} beam predictions to results_beam10.csv")

api.competition_submit(
    file_name   = "results_beam10.csv",
    message     = "HW4P2 beam-10 submission",
    competition = "11785-hw-4-p-2-asr-with-transformer-spring-2026"
)

# Autolab Code Submission

Follow the steps below to package and submit your code.

In [ ]:
# Step 1: Assign your final trained model
MODEL = model  # Must be the exact model used for your best Kaggle submission

In [ ]:
# Step 2: Fill in README
README = """
- **Model**: EncoderDecoderTransformer with 4 encoder layers + 2 decoder layers,
  d_model=256, 4 heads, d_ff=1024. Conv2D subsampling with time_reduction=4.
  Char tokenization (vocab ~35). CTC auxiliary loss weight=0.1.
- **Training Strategy**: AdamW (lr=3e-4, betas=(0.9,0.98)), CosineAnnealingLR
  with 3-epoch linear warmup. Label smoothing=0.1. Gradient accumulation steps=2.
  Trained for 30 epochs.
- **Augmentations**: SpecAugment — 2 freq masks (width≤27) + 2 time masks (width≤100).
  Global MVN normalization.
- **Notebook Execution**: Run cells top-to-bottom. Set the correct data root in
  config.yaml before running dataset cells. GPU required for training.
"""

In [ ]:
# Step 3: Credentials
KAGGLE_USERNAME            = "yufeison"
KAGGLE_API_KEY             = "KGAT_0e2401c4d351c4ce89b30f0778b2f306"
WANDB_API_KEY              = ""  # TODO: Add WandB key if using WandB
WANDB_USERNAME_OR_TEAMNAME = ""  # TODO: WandB username
WANDB_PROJECT              = "hw4p2"

In [ ]:
# Step 4: Set notebook path
NOTEBOOK_PATH = "/kaggle/working/IDL-HW4/HW4P2_notebook.ipynb"  # TODO: Set correct path

In [ ]:
# Step 5: Additional files
ADDITIONAL_FILES = ["config.yaml"]

In [ ]:
# Step 6: Generate submission zip
ACKNOWLEDGED = True  # Set True after reading all requirements

!git clone https://github.com/CMU-IDeeL/S26-HWP2-Submission-Backend.git
!mv S26-HWP2-Submission-Backend/submission .
!rm -rf S26-HWP2-Submission-Backend

from submission.submission_config import SubmissionConfig
from submission.backend_config import BackendConfig, HW4P2_BACKEND_CONFIG
from submission.main import create_submission_zip

create_submission_zip(
    cfg = SubmissionConfig(
        model                      = MODEL,
        kaggle_username            = KAGGLE_USERNAME,
        kaggle_api_key             = KAGGLE_API_KEY,
        wandb_api_key              = WANDB_API_KEY,
        wandb_username_or_teamname = WANDB_USERNAME_OR_TEAMNAME,
        wandb_project              = WANDB_PROJECT,
        notebook_path              = NOTEBOOK_PATH,
        additional_files           = ADDITIONAL_FILES,
        readme                     = README,
        acknowledged               = ACKNOWLEDGED,
    ),
    backend_cfg = HW4P2_BACKEND_CONFIG
)